# 07 · BM25 and hybrid search

Embeddings are good at capturing paraphrasing, but they may miss exact code, metric names, or identifiers. BM25 evaluates term coverage with corrections for frequency and document length. We combine the BM25 and dense rankings using **Reciprocal Rank Fusion (RRF)**, rather than by adding incomparable raw scores.

In [ ]:
# Only cell with imports and project path.
import math
import re
import sys
from pathlib import Path

import numpy as np
from rank_bm25 import BM25Okapi

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, section_records

## 1. How is the BM25 score calculated?

For a term `t` and a document `d`: `IDF(t) × tf(t,d)(k1+1) / [tf(t,d)+k1(1-b+b·|d|/avgdl)]`. `tf` saturates: the tenth occurrence yields less than the first. `b` regulates the length penalty, `k1` handles saturation. A common term has a lower IDF. The `rank_bm25.BM25Okapi` version applies its own IDF smoothing, so the calculation below is an intuition, not a guarantee of an identical number.

In [ ]:
toy = ["MASE uses a training denominator", "Percentage error can have zero denominator", "A forecast is frozen before the next month"]
tokens = [re.findall("[a-z0-9]+", text.lower()) for text in toy]
term = "denominator"
N = len(tokens)
df = sum(term in row for row in tokens)
idf = math.log(1 + (N - df + 0.5) / (df + 0.5))
{"df": df, "N": N, "illustrative_idf": round(idf, 3),
 "bm25_package_scores": BM25Okapi(tokens).get_scores([term]).round(3).tolist()}

Observe the two mechanisms separately at a constant `IDF`: repetitions of a term yield increasingly smaller gains, while a longer segment with the same single hit receives a lower score. This is the calculation from the provided formula, so the values may differ from the smoothed implementation in the library.

In [ ]:
k1, b, average_length = 1.5, 0.75, 100
frequency_effect = [(tf, round(idf * tf * (k1 + 1) /
                               (tf + k1 * (1 - b + b * 100 / average_length)), 3))
                    for tf in (1, 2, 4, 8)]
length_effect = [(length, round(idf * (k1 + 1) /
                                (1 + k1 * (1 - b + b * length / average_length)), 3))
                 for length in (50, 100, 200)]
{"tf_repetitions": frequency_effect, "length_at_tf_1": length_effect}

## 2. Two rankings of the same sections

We perform the tokenization of the query and documents using the same rule. This simple tokenizer is sufficient for English educational documents, but it does not support Polish declension, phrases, or special identifiers. The dense model works on the original text.

In [ ]:
records = section_records()
terms = [re.findall("[a-z0-9]+", row["text"].lower()) for row in records]
bm25 = BM25Okapi(terms)
matrix = embed_texts([row["text"] for row in records])
matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)
question = "Why can MASE have an undefined training denominator?"
lexical = bm25.get_scores(re.findall("[a-z0-9]+", question.lower()))
q = embed_texts([question])[0]
q /= np.linalg.norm(q)
dense = matrix @ q
bm_rank = np.argsort(-lexical)[:8]
vec_rank = np.argsort(-dense)[:8]
{"BM25": [(records[int(i)]["id"], round(float(lexical[i]), 2)) for i in bm_rank[:4]],
 "dense": [(records[int(i)]["id"], round(float(dense[i]), 3)) for i in vec_rank[:4]]}

## 3. Reciprocal Rank Fusion

`RRF(d) = Σ 1/(k+rank(d))`; we use the rank, not the BM25 or cosine values. `k=60` mitigates the impact of a single first-place position. A candidate outside the top-8 of a given source contributes no points from that source. Then, the top-k can be passed to a reranker.

In [ ]:
rrf = np.zeros(len(records))
for ranking in (bm_rank, vec_rank):
    for rank, index in enumerate(ranking, 1):
        rrf[index] += 1 / (60 + rank)
hybrid_rank = np.argsort(-rrf)[:6]
[{"id": records[int(i)]["id"], "heading": records[int(i)]["heading"],
  "rrf": round(float(rrf[i]), 4), "bm25": round(float(lexical[i]), 2),
  "cosine": round(float(dense[i]), 3)} for i in hybrid_rank]

## 4. When does hybrid help, and when does it harm?

BM25 helps with `MASE`, product codes, and specific numbers. Dense helps with descriptive questions and paraphrases. Hybrid can worsen the result when a poorly tokenized common word pushes out a relevant paraphrase. The access filter should limit **both** candidate sets before fusion. Check recall@k on labeled questions; ranking is not truth.

Check two descriptive questions: one about zero demand, the other about the impact of out-of-stock items on the backtest. Compare the position of the correct section in each ranking. RRF can move it up **or** down; results depend on the corpus and the embedding model.

In [ ]:
cases = [
    ("How can we assess forecast accuracy when some periods have zero demand?", "SCI-FORECAST-2011:v1:s04"),
    ("How should stockout effects be handled in forecast backtests?", "SCI-STOCK-2011:v1:s04"),
]
case_vectors = embed_texts([text for text, _ in cases])
case_vectors /= np.linalg.norm(case_vectors, axis=1, keepdims=True)
comparisons = []
for (text, target_id), query_vector in zip(cases, case_vectors):
    lexical_scores = bm25.get_scores(re.findall("[a-z0-9]+", text.lower()))
    dense_scores = matrix @ query_vector
    bm_order, dense_order = np.argsort(-lexical_scores), np.argsort(-dense_scores)
    fused_scores = np.zeros(len(records))
    for ranking in (bm_order[:8], dense_order[:8]):
        for rank, index in enumerate(ranking, 1):
            fused_scores[index] += 1 / (60 + rank)
    target = next(i for i, row in enumerate(records) if row["id"] == target_id)
    comparisons.append({"query": text, "target": target_id,
        "rank_bm25": int(np.flatnonzero(bm_order == target)[0] + 1),
        "rank_dense": int(np.flatnonzero(dense_order == target)[0] + 1),
        "rank_hybrid": int(np.flatnonzero(np.argsort(-fused_scores) == target)[0] + 1)})
comparisons

Target position is only one metric. For a larger set of queries, also mark all fragments sufficient to answer the question and compare recall@k or nDCG@k. **Exercise:** ask for the exact document identifier and evaluate whether tokenization breaks it correctly. The [BM25 library](https://github.com/dorianbrown/rank_bm25) also explains that it does not tokenize text itself.